# Project 2: Smart Vision System – Modular Implementation

This notebook has been **modularized** into separate Python files for better organization and reusability. The original functionality is now available as a complete pipeline with improved error handling and documentation.

Target platform: **Raspberry Pi 5 (64‑bit Bookworm)** + **AI Camera** + *(optional)* **AI HAT (Hailo‑8/8L)**.

## 🚀 **NEW: Modular Python Implementation**

The code has been reorganized into the following modules:

### **Core Modules:**
- **`setup_environment.py`** - Environment setup and package installation
- **`model_manager.py`** - TensorFlow Lite model download and management  
- **`object_detector.py`** - Object detection with manual/auto modes
- **`statistical_analysis.py`** - Week 6-8 statistical analysis
- **`main_detection.py`** - Complete pipeline orchestration

### **Supporting Files:**
- **`requirements.txt`** - Python dependencies
- **`README.md`** - Comprehensive documentation

## 📋 **Quick Start Guide**

### **Option 1: Use the Modular Python Files (Recommended)**

```bash
# 1. Setup environment
python3 setup_environment.py

# 2. Run detection (manual mode)
python3 main_detection.py --mode manual --light-level high --distance 15.0 --ground-truth person

# 3. Run detection (auto mode) 
python3 main_detection.py --mode auto --frames 100 --interval 0.2 --light-level low --ground-truth car

# 4. Run with statistical analysis
python3 main_detection.py --mode auto --frames 50 --analyze
```

### **Option 2: Use Original Notebook Cells (Below)**

The original notebook cells are preserved below for reference and individual testing.

## 🔧 **What the modular system provides:**
1. **Improved Error Handling** - Better error messages and recovery
2. **Command Line Interface** - Easy parameter configuration
3. **Modular Design** - Use individual components as needed
4. **Better Documentation** - Comprehensive README and docstrings
5. **Quality Control** - Data validation and cleaning features

> **Tip**: The modular implementation fixes several issues found in the original notebook and provides a more robust, production-ready system.


## 🔧 **System Setup (Terminal Commands)**

**Run these commands in a terminal before using the Python modules:**

### **Basic Setup:**
```bash
# Update system and install camera support
sudo apt update && sudo apt full-upgrade -y
sudo apt install -y python3-picamera2 rpicam-apps

# Test camera functionality
rpicam-hello -t 3000
```

### **Optional: AI HAT (Hailo) Setup:**
```bash
# Install Hailo support
sudo apt install -y hailo-all
sudo reboot

# Verify Hailo installation
hailortcli scan
rpicam-hello -t 0 --post-process-file /usr/share/rpi-camera-assets/hailo_yolov6_inference.json
```

### **Python Environment Setup:**
```bash
# Install Python packages (automated)
python3 setup_environment.py

# Or install manually
pip install -r requirements.txt
```


## 📦 **Python Package Installation**

**Use the modular approach (recommended):**
```bash
python3 setup_environment.py
```

**Or run this cell for in-notebook installation:**
> If you prefer a venv, create it in terminal and start Jupyter from that venv. Otherwise, this cell will install into the user environment.


In [ ]:
import sys, subprocess, pkgutil
def pip_install(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade'] + pkgs)

need = ['numpy', 'pandas', 'matplotlib', 'scipy', 'opencv-python', 'tflite-runtime', 'picamera2']
to_install = [p for p in need if not pkgutil.find_loader(p.split('==')[0])]
if to_install:
    print('Installing:', to_install)
    try:
        pip_install(to_install)
    except Exception as e:
        print('If Picamera2 fails via pip, install system package: sudo apt install -y python3-picamera2')
        raise e
else:
    print('All packages already installed')


## 🤖 **Model Download**

**Use the modular approach (recommended):**
```bash
python3 model_manager.py
```

**Or run this cell for in-notebook model download:**
This fetches the classic COCO SSD MobileNet v1 quantized model. If the URL changes, replace it or copy your own `.tflite` + `labelmap.txt`.


In [ ]:
import os, urllib.request, zipfile
os.makedirs('models', exist_ok=True)
url = 'https://storage.googleapis.com/download.tensorflow.org/models/tflite/coco_ssd_mobilenet_v1_1.0_quant_2018_06_29.zip'
zip_path = 'models/coco_ssd.zip'
if not os.path.exists('models/detect.tflite'):
    print('Downloading model...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('models')
    print('Unzipped to models/.')
else:
    print('Model already present.')
print('models directory:', os.listdir('models'))


## 📷 **Camera Test**

**Use the modular approach (recommended):**
```bash
python3 setup_environment.py  # Includes camera test
```

**Or run this cell for in-notebook camera test:**
Captures a single frame and shows its shape (no preview needed for headless).


In [ ]:
from picamera2 import Picamera2
import numpy as np

picam2 = Picamera2()
config = picam2.create_preview_configuration(main={'size': (640,480), 'format': 'RGB888'})
picam2.configure(config)
picam2.start()
frame = picam2.capture_array()
picam2.stop()
frame.shape


## 🎯 **Object Detection & Logging**

**Use the modular approach (recommended):**

### **Manual Mode (Press Enter for each frame):**
```bash
python3 main_detection.py --mode manual --light-level high --distance 15.0 --ground-truth person
```

### **Automatic Mode (Timed capture):**
```bash
python3 main_detection.py --mode auto --frames 100 --interval 0.2 --light-level low --ground-truth car
```

### **With Statistical Analysis:**
```bash
python3 main_detection.py --mode auto --frames 50 --analyze
```

**Or run this cell for in-notebook detection:**
Two modes:
- **Manual**: press Enter per frame for controlled labeling.
- **Auto**: capture N frames at a fixed interval.

You must provide the **lighting label** (`low|medium|high`), optional **distance** (cm), and the **ground truth** label (what you placed in view) for `is_correct`.


In [ ]:
import os, csv, time
from datetime import datetime
import cv2
import numpy as np
from picamera2 import Picamera2
from tflite_runtime.interpreter import Interpreter

MODEL_PATH = 'models/detect.tflite'
# Some archives provide labelmap.txt or coco_labels.txt; detect and pick one
LABEL_PATHS = [
    'models/labelmap.txt', 'models/coco_labels.txt', 'models/labelmap.txt'
]
LABEL_PATH = next((p for p in LABEL_PATHS if os.path.exists(p)), None)
CSV_PATH = 'run_log.csv'

def load_labels(path):
    if path is None:
        # fallback COCO 90 labels truncated (placeholder)
        return [f'class_{i}' for i in range(91)]
    return [l.strip() for l in open(path, 'r').readlines()]

labels = load_labels(LABEL_PATH)
interp = Interpreter(model_path=MODEL_PATH)
interp.allocate_tensors()
in_det = interp.get_input_details()
out_det = interp.get_output_details()
ih, iw = in_det[0]['shape'][1], in_det[0]['shape'][2]

def ensure_csv(path=CSV_PATH):
    new = not os.path.exists(path)
    f = open(path, 'a', newline='')
    w = csv.writer(f)
    if new:
        w.writerow(['timestamp','predicted_label','is_correct','confidence',
                    'inference_time_ms','light_level','distance_cm'])
    return f, w

def infer_frame(rgb):
    rgb_in = cv2.resize(rgb, (iw, ih))
    rgb_in = np.expand_dims(rgb_in, 0).astype(np.uint8)
    t0 = time.perf_counter()
    interp.set_tensor(in_det[0]['index'], rgb_in)
    interp.invoke()
    dt_ms = (time.perf_counter() - t0) * 1000.0
    boxes   = interp.get_tensor(out_det[0]['index'])[0]
    classes = interp.get_tensor(out_det[1]['index'])[0].astype(int)
    scores  = interp.get_tensor(out_det[2]['index'])[0]
    if len(scores) and float(np.max(scores)) > 0:
        k = int(np.argmax(scores))
        cls = classes[k]
        label = labels[cls] if cls < len(labels) else f'class_{cls}'
        conf  = float(scores[k])
    else:
        label, conf = 'none', 0.0
    return label, conf, dt_ms

def run_manual(light_level='unknown', distance_cm=np.nan, ground_truth=None):
    picam2 = Picamera2()
    picam2.configure(picam2.create_preview_configuration(main={'size': (640,480),'format':'RGB888'}))
    picam2.start()
    f, w = ensure_csv()
    try:
        print('Press Enter to capture; Ctrl+C to stop')
        while True:
            input()
            frame = picam2.capture_array()
            label, conf, dt = infer_frame(frame)
            is_correct = int(ground_truth is not None and label == ground_truth)
            w.writerow([datetime.now().isoformat(timespec='seconds'), label, is_correct,
                        round(conf,4), round(dt,2), light_level, distance_cm])
            f.flush()
            print(f'label={label} conf={conf:.2f} {dt:.1f}ms saved')
    except KeyboardInterrupt:
        pass
    finally:
        f.close(); picam2.stop()

def run_auto(n_frames=100, interval_sec=0.2, light_level='unknown', distance_cm=np.nan, ground_truth=None):
    picam2 = Picamera2()
    picam2.configure(picam2.create_preview_configuration(main={'size': (640,480),'format':'RGB888'}))
    picam2.start()
    f, w = ensure_csv()
    try:
        for i in range(n_frames):
            frame = picam2.capture_array()
            label, conf, dt = infer_frame(frame)
            is_correct = int(ground_truth is not None and label == ground_truth)
            w.writerow([datetime.now().isoformat(timespec='seconds'), label, is_correct,
                        round(conf,4), round(dt,2), light_level, distance_cm])
            f.flush()
            print(f'[{i+1}/{n_frames}] label={label} conf={conf:.2f} {dt:.1f}ms saved')
            time.sleep(interval_sec)
    finally:
        f.close(); picam2.stop()

print('Functions defined: run_manual(), run_auto()')
print('Examples:')
print("run_manual('high', 15.0, 'square')  # press Enter per frame")
print("run_auto(120, 0.2, 'low', 10.0, 'circle')  # capture 120 frames")


## 📊 **Week 6 – Discrete Distribution Analysis**

**Use the modular approach (recommended):**
```bash
python3 statistical_analysis.py  # Runs all analyses
```

**Or run this cell for in-notebook analysis:**
Load `run_log.csv` and compute/fit distributions for `is_correct` windows, attempts to first success, and counts per time window.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv('run_log.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(df.head())

# --- Binomial: number of correct per K images ---
K = 10
bins = [df['is_correct'][i:i+K].sum() for i in range(0, len(df), K) if len(df)-i >= K]
p_hat = np.mean(df['is_correct'])
x = np.arange(0, K+1)
pmf = stats.binom.pmf(x, K, p_hat)

plt.figure();
plt.hist(bins, bins=range(0,K+2), density=True, alpha=0.6, label='empirical')
plt.plot(x, pmf, marker='o', label=f'Binom(n={K}, p={p_hat:.2f})')
plt.xlabel('# correct in window of 10'); plt.ylabel('Probability'); plt.legend(); plt.title('Binomial fit')
plt.show()

# --- Geometric: trials until first success (per sequence) ---
def trials_until_success(series):
    c = 0
    for v in series:
        c += 1
        if v == 1:
            return c
    return np.nan

geom_samples = []
for i in range(0, len(df), K):
    chunk = df['is_correct'][i:i+K].tolist()
    if len(chunk) == K:
        geom_samples.append(trials_until_success(chunk))
geom_samples = [g for g in geom_samples if not np.isnan(g)]
p_geo = 1.0/np.mean(geom_samples) if geom_samples else np.nan

plt.figure();
plt.hist(geom_samples, bins=range(1,K+2), density=True, alpha=0.6, label='empirical')
gx = np.arange(1, K+1)
gpmf = stats.geom.pmf(gx, p_geo) if not np.isnan(p_geo) else np.zeros_like(gx)
plt.plot(gx, gpmf, marker='o', label=f'Geom(p~{p_geo:.2f})')
plt.xlabel('Trials until first success'); plt.ylabel('Probability'); plt.legend(); plt.title('Geometric fit')
plt.show()

# --- Poisson: detections per fixed time window ---
df['sec'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
W = 10  # seconds
df['bucket'] = (df['sec']//W).astype(int)
counts = df.groupby('bucket')['is_correct'].sum()  # using successes as "detections"
lam = counts.mean() if len(counts) else np.nan
px = np.arange(0, max(5,int(counts.max())+2) if len(counts) else 6)
ppmf = stats.poisson.pmf(px, lam) if not np.isnan(lam) else np.zeros_like(px)

plt.figure();
plt.hist(counts, bins=range(int(counts.max()+2) if len(counts) else 2), density=True, alpha=0.6, label='empirical')
plt.plot(px, ppmf, marker='o', label=f'Poisson(λ~{lam:.2f})')
plt.xlabel(f'# successes per {W}s window'); plt.ylabel('Probability'); plt.legend(); plt.title('Poisson fit')
plt.show()


## 📈 **Week 7 – Continuous Distribution Modeling**

**Use the modular approach (recommended):**
```bash
python3 statistical_analysis.py  # Includes AIC/BIC analysis
```

**Or run this cell for in-notebook analysis:**
Fits to: confidence → Beta/Normal; inference time → Exponential/Weibull; distance → Gaussian/Lognormal (if available).


In [ ]:
def aic(logL, k):
    return 2*k - 2*logL
def bic(logL, k, n):
    return k*np.log(n) - 2*logL

def fit_and_compare(data, cand):
    data = np.asarray(data)
    data = data[np.isfinite(data)]
    n = len(data)
    results = []
    for name, dist in cand:
        params = dist.fit(data)
        logpdf = dist.logpdf(data, *params)
        logL = float(np.sum(logpdf))
        k = len(params)
        results.append((name, params, aic(logL,k), bic(logL,k,n)))
    return sorted(results, key=lambda x: x[2])  # sort by AIC

conf = df['confidence'].clip(1e-6, 1-1e-6)
tms  = df['inference_time_ms'].clip(1e-6)
dist = df['distance_cm'].dropna() if 'distance_cm' in df else pd.Series(dtype=float)

cand_conf = [('Beta', stats.beta), ('Normal', stats.norm)]
cand_time = [('Exponential', stats.expon), ('Weibull', stats.weibull_min)]
cand_dist = [('Gaussian', stats.norm), ('Lognormal', stats.lognorm)] if len(dist)>0 else []

print('Confidence fit:')
print(fit_and_compare(conf, cand_conf))
print('\nInference time fit:')
print(fit_and_compare(tms,  cand_time))
if cand_dist:
    print('\nDistance fit:')
    print(fit_and_compare(dist, cand_dist))

# Visuals
def plot_fit(data, dist, params, title, bins=30, is_cdf=False):
    data = np.asarray(data)
    plt.figure()
    if is_cdf:
        xs = np.linspace(np.min(data), np.max(data), 200)
        plt.plot(np.sort(data), np.arange(1,len(data)+1)/len(data), label='Empirical CDF')
        plt.plot(xs, dist.cdf(xs, *params), label=f'{dist.name} CDF')
        plt.ylabel('Cumulative probability')
    else:
        plt.hist(data, bins=bins, density=True, alpha=0.6, label='Empirical')
        xs = np.linspace(np.min(data), np.max(data), 200)
        plt.plot(xs, dist.pdf(xs, *params), label=f'{dist.name} PDF')
        plt.ylabel('Density')
    plt.xlabel('Value'); plt.title(title); plt.legend(); plt.show()

# Example: plot best for inference time
best_time = fit_and_compare(tms, cand_time)[0]
name, params, AIC, BIC = best_time
dmap = {'Exponential': stats.expon, 'Weibull': stats.weibull_min}
plot_fit(tms, dmap[name], params, f'Inference time – best fit: {name}', bins=30, is_cdf=False)
plot_fit(tms, dmap[name], params, f'Inference time CDF – best fit: {name}', bins=30, is_cdf=True)


## 🔗 **Week 8 – Joint Distributions & Correlations**

**Use the modular approach (recommended):**
```bash
python3 statistical_analysis.py  # Includes joint analysis
```

**Or run this cell for in-notebook analysis:**
Scatter + marginals and some conditional/joint probability estimates.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Correlations
pairs = []
if 'distance_cm' in df:
    pairs.append(('confidence','distance_cm'))
pairs.append(('inference_time_ms','is_correct'))

for a,b in pairs:
    sub = df[[a,b]].dropna()
    if len(sub) > 2:
        corr = np.corrcoef(sub[a], sub[b])[0,1]
        print(f'Correlation({a}, {b}) = {corr:.3f}')

# Scatter with marginal hist for confidence vs. inference time
sns.jointplot(data=df, x='confidence', y='inference_time_ms', kind='scatter', marginal_kws={'bins':30})
plt.suptitle('Confidence vs Inference Time (with marginals)'); plt.tight_layout(); plt.show()

# Conditional probabilities examples
p_succ_conf08 = np.mean((df['is_correct']==1) & (df['confidence']>0.8))
p_conf09_low  = np.nan
if 'light_level' in df.columns:
    low = df[df['light_level'].astype(str).str.lower()=='low']
    if len(low)>0:
        p_conf09_low = np.mean(low['confidence']>0.9)
print('P(Success and Confidence>0.8) =', p_succ_conf08)
print('P(Confidence>0.9 | Lighting=Low) =', p_conf09_low)


## 📁 **Export Cleaned Data**

**Use the modular approach (recommended):**
```bash
python3 statistical_analysis.py  # Automatically exports cleaned data
```

**Or run this cell for in-notebook export:**
Saves a copy with basic QC flags and ordering.


In [ ]:
dfc = df.copy()
dfc = dfc.sort_values('timestamp')
fps_est = 1000.0/dfc['inference_time_ms'].replace(0,np.nan)
dfc['qc_flag'] = ((fps_est > fps_est.median()+2*fps_est.std()) & (dfc['is_correct']==0)).astype(int)
dfc.to_csv('run_log_clean.csv', index=False)
print('Saved run_log_clean.csv')


## 🚀 **Quick Reference Guide**

### **Modular Approach (Recommended):**

```bash
# 1. Setup environment and test camera
python3 setup_environment.py

# 2. Download model
python3 model_manager.py

# 3. Run detection (choose one):
# Manual mode
python3 main_detection.py --mode manual --light-level high --distance 15.0 --ground-truth person

# Auto mode
python3 main_detection.py --mode auto --frames 100 --interval 0.2 --light-level low --ground-truth car

# 4. Run statistical analysis
python3 statistical_analysis.py
```

### **Original Notebook Approach:**

1. Confirm camera works: `rpicam-hello -t 3000` (terminal).
2. (Optional) Install Hailo: `sudo apt install -y hailo-all && sudo reboot`.
3. Run Section **1** (packages) and **2** (download model).
4. Run Section **3** (camera test).
5. Use **run_manual** or **run_auto** to create `run_log.csv`.
6. Run Sections **5–8** to analyze your data and export `run_log_clean.csv`.

### **Command Line Options:**
```bash
python3 main_detection.py --help  # See all available options
```
